# Week 7 · Deep dive (optional): Microsoft Agent Framework, in depth

**Optional and not graded.** Do the [lab](lab.ipynb) first: it ships `/v1/resolve`. This notebook takes Agent Framework apart, one piece at a time, following the [lecture notes](lecture.md). Most parts are small programs about the same support case.

| Part | What you explore | Lecture notes |
|---|---|---|
| 1 | A workflow from scratch: executors, edges, events, a diagram | §5 |
| 2 | Fan-out and fan-in without a model: supersteps | §5, §6 |
| 3 | Conditional routing: switch-case edges driven by a structured triage agent | §6 |
| 4 | Middleware: a policy the model cannot talk its way around | §4, §10 |
| 5 | A person in the loop that survives a restart: requests and checkpoints | §7 |
| 6 | Orchestration patterns: group chat and handoff | §8 |
| 7 | Agents as tools, and what a team costs | §8, §11 |
| 8 | Evaluating an agent with `evaluate_agent` | §12 |

Open it after you have submitted your URL: it does not edit `app/` or deploy, so it cannot change your score. Run the setup cell, then the parts in order. Parts 2, 3 and 5 reuse imports from Parts 1 and 2, so a part run on its own can fail with a `NameError`. You can stop after any part. Cost: a few cents of model calls.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.py").exists())))
import course

course.setup()

In [ ]:
from agent_framework import Agent

from app import domain
from app.config import get_settings
from app.state import AppState
from app.tools import get_station, issue_credit, quote_fare, recording, search_handbook

state = AppState(get_settings())
client = state.models.agent_client()  # one FoundryChatClient, shared by every agent below

CASE = (
    "Rider R-1042 is a member. They took a 40-minute e-bike ride from LS-14; the battery died halfway "
    "and they pushed the bike to the dock. They want the e-bike charge back, and the bike should be checked."
)

## Part 1 · A workflow from scratch

A **workflow** is a graph you define in code. Its nodes are **executors**: units that receive a typed message and either send a message on or yield an output. Its arrows are **edges**. Agent Framework checks at build time that each edge connects compatible types.

Two ways to write an executor: a class with a `@handler` method, or a function with `@executor`. The type parameters of `WorkflowContext[Send, Yield]` say what the executor sends on and what it may yield as workflow output. No model is involved yet.

In [ ]:
import re

from agent_framework import Executor, WorkflowBuilder, WorkflowContext, WorkflowViz, executor, handler
from typing_extensions import Never


class Normalize(Executor):
    @handler
    async def run(self, case: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(" ".join(case.split()))  # sends a str to the next executor


@executor(id="extract_station")
async def extract_station(case: str, ctx: WorkflowContext[Never, str]) -> None:
    found = re.findall(r"\bLS-\d{2}\b", case)
    await ctx.yield_output(found[0] if found else "none")  # yields the workflow's output


normalize = Normalize(id="normalize")
workflow = WorkflowBuilder(start_executor=normalize, name="station-extractor").add_edge(normalize, extract_station).build()

result = await workflow.run("  battery died   at LS-14 \n")
print("output:", result.get_outputs())
print(WorkflowViz(workflow).to_mermaid())

Paste the Mermaid text into any Mermaid renderer (GitHub renders it in Markdown) to see the graph. Now watch the run as a stream of **events**: this is what you would forward to a UI or a log.

In [ ]:
async for event in workflow.run("dock broken at LS-03", stream=True):
    print(f"{event.type:24s} {getattr(event, 'executor_id', '') or ''}")

## Part 2 · Fan-out and fan-in without a model

Fan-out sends one message to several executors; fan-in waits for all of them and hands the join a **list**. Executors that receive messages in the same step run concurrently, in one **superstep**; the next superstep starts when they all finish. Gathering facts in parallel is plain Python here, so the result is deterministic.

In [ ]:
from dataclasses import dataclass


@dataclass
class Fact:
    source: str
    value: str


@executor(id="split")
async def split(case: str, ctx: WorkflowContext[str]) -> None:
    await ctx.send_message(case)


@executor(id="fare")
async def fare(case: str, ctx: WorkflowContext[Fact]) -> None:
    await ctx.send_message(Fact("fare", str(domain.quote_fare("member", 40, True))))


@executor(id="station")
async def station(case: str, ctx: WorkflowContext[Fact]) -> None:
    await ctx.send_message(Fact("station", str(domain.get_station("LS-14"))))


@executor(id="join")
async def join(facts: list[Fact], ctx: WorkflowContext[Never, dict]) -> None:
    await ctx.yield_output({f.source: f.value for f in facts})


fan = (
    WorkflowBuilder(start_executor=split, name="gather-facts")
    .add_fan_out_edges(split, [fare, station])
    .add_fan_in_edges([fare, station], join)
    .build()
)
for key, value in (await fan.run(CASE)).get_outputs()[0].items():
    print(f"{key:8s} {value}")
print()
async for event in fan.run(CASE, stream=True):
    if event.type in ("superstep_started", "executor_invoked"):
        print(f"{event.type:20s} {getattr(event, 'executor_id', '') or ''}")

`fare` and `station` are invoked in the same superstep; `join` runs in the next one with both facts. This is the shape of the lab's starting `resolve` service (all specialists in parallel, then a writer), minus the models.

## Part 3 · Conditional routing with switch-case edges

In the lab you wrote triage in Python: call a triage agent, then decide whom to call. A workflow can express that as **edges with conditions**. The triage executor returns a **structured** decision (a `Literal` field, so the model cannot invent a fourth team) and the graph routes on it.

In [ ]:
from typing import Literal

from agent_framework import Case, Default
from pydantic import BaseModel


class TriageDecision(BaseModel):
    owner: Literal["billing", "operations", "policy"]
    reason: str


@dataclass
class Routed:
    case: str
    decision: TriageDecision


triage_agent = Agent(
    client=client,
    name="triage",
    instructions="Pick the ONE team that owns this Lakeshore Ride case: billing (money owed, credits), "
    "operations (broken bikes, docks, stations), policy (rules and questions). Explain in one sentence.",
)


class Triage(Executor):
    @handler
    async def run(self, case: str, ctx: WorkflowContext[Routed]) -> None:
        decision = (await triage_agent.run(case, options={"response_format": TriageDecision})).value
        await ctx.send_message(Routed(case, decision))


class Specialist(Executor):
    def __init__(self, name: str, instructions: str, tools: list):
        super().__init__(id=name)
        self.agent = Agent(client=client, name=name, instructions=instructions, tools=tools)

    @handler
    async def run(self, routed: Routed, ctx: WorkflowContext[Never, str]) -> None:
        reply = await self.agent.run(routed.case)
        await ctx.yield_output(f"[{self.id}] because: {routed.decision.reason}\n  {reply.text}")


triage = Triage(id="triage")
billing = Specialist("billing", "You handle money. Compute fares with quote_fare. Recommend credit; never issue it.", [quote_fare, search_handbook])
operations = Specialist("operations", "You handle stations and bikes. Say what should be repaired.", [get_station])
policy = Specialist("policy", "Explain the relevant handbook rule in two sentences.", [search_handbook])

router = (
    WorkflowBuilder(start_executor=triage, name="triage-router")
    .add_switch_case_edge_group(
        triage,
        [
            Case(condition=lambda r: r.decision.owner == "billing", target=billing),
            Case(condition=lambda r: r.decision.owner == "operations", target=operations),
            Default(target=policy),
        ],
    )
    .build()
)
with recording(state):
    for case in [CASE, "The dock at LS-10 will not lock.", "Can a 15-year-old ride with a parent?"]:
        print((await router.run(case)).get_outputs()[0], "\n")
print(WorkflowViz(router).to_mermaid())

The first case belongs to two teams (billing *and* operations), but a switch-case sends it to exactly one. For "one or more", use `add_multi_selection_edge_group`, whose selection function returns a list of targets. Which shape fits the lab's `/v1/resolve`, where `w07.resolve.route` rewards "the right specialists, and no others"?

## Part 4 · Middleware: a policy the model cannot talk around

Instructions are requests; middleware is code. A **function middleware** runs around every tool call, sees the function name and the arguments, and may refuse to call it. Here the rule "credits above $25 need a supervisor" lives in code the model cannot skip:

In [ ]:
from agent_framework import FunctionInvocationContext, function_middleware


@function_middleware
async def cap_credits(context: FunctionInvocationContext, call_next) -> None:
    if context.function.name == "issue_credit" and float(context.arguments["amount"]) > domain.CREDIT_LIMIT:
        print("  middleware refused:", dict(context.arguments))
        context.result = f"Refused by policy: credits above ${domain.CREDIT_LIMIT:.0f} need a supervisor."
        return  # not calling call_next() means the tool never runs
    await call_next()


generous = Agent(
    client=client,
    name="billing",
    instructions="Issue the credit the user asks for with issue_credit, then report the result.",
    tools=[issue_credit],
    middleware=[cap_credits],
)
for ask in ["Please credit rider R-1042 $60 for the trouble.", "Please credit rider R-1042 $7.20 for the dead battery."]:
    with recording(state) as calls:
        reply = await generous.run(ask)
    print(ask, "\n  tool actually ran:", [c["arguments"] for c in calls], "\n  reply:", reply.text[:160], "\n")

The model *asked* for $60; the middleware refused, so the recording shows no call for the first request. The model then explains the refusal, because the refusal came back as the tool result. Agent Framework also has **agent middleware** (around a whole run) and **chat middleware** (around each model call); Part 7 uses the latter to count tokens.

## Part 5 · A person in the loop that survives a restart

The lab's approval step happens within one process. Real approvals take hours: the process may restart before the supervisor answers. A workflow can **pause** with a request (`ctx.request_info`), save its state in a **checkpoint**, and resume later in a different process from that checkpoint.

The request type is defined at module level and registered as an allowed checkpoint type: checkpoints are serialized, and the storage only restores types you allow.

In [ ]:
import tempfile

from agent_framework import FileCheckpointStorage, response_handler


@dataclass
class CreditRequest:
    rider_id: str
    amount: float


class Approver(Executor):
    @handler
    async def run(self, request: CreditRequest, ctx: WorkflowContext[Never, dict]) -> None:
        if request.amount <= domain.CREDIT_LIMIT:
            await ctx.yield_output(domain.issue_credit(request.rider_id, request.amount, "within agent limit"))
            return
        await ctx.request_info(request, bool)  # pause: ask a person, expect a bool back

    @response_handler
    async def on_decision(self, original: CreditRequest, approved: bool, ctx: WorkflowContext[Never, dict]) -> None:
        status = "issued_by_supervisor" if approved else "rejected_by_supervisor"
        await ctx.yield_output({"status": status, "rider_id": original.rider_id, "amount": original.amount})


def build_approval(storage: FileCheckpointStorage):
    return WorkflowBuilder(start_executor=Approver(id="approver"), name="credit-approval", checkpoint_storage=storage).build()


folder = tempfile.mkdtemp(prefix="checkpoints-")
ALLOWED = ["__main__:CreditRequest"]
approval = build_approval(FileCheckpointStorage(folder, allowed_checkpoint_types=ALLOWED))

print("small credit:", (await approval.run(CreditRequest("R-1042", 7.2))).get_outputs())
paused = await approval.run(CreditRequest("R-2210", 60.0))
print("large credit outputs:", paused.get_outputs())
print("waiting for a person:", [(r.request_id[:8], r.data) for r in paused.get_request_info_events()])
print("checkpoint files on disk:", len(list(Path(folder).glob("*.json"))))

Now simulate a restart: throw away the workflow object, build a new one over the same folder, restore the latest checkpoint, and answer the pending request.

In [ ]:
del approval  # the "process" is gone

storage = FileCheckpointStorage(folder, allowed_checkpoint_types=ALLOWED)
approval = build_approval(storage)
latest = max(await storage.list_checkpoints(workflow_name="credit-approval"), key=lambda c: c.timestamp)
restored = await approval.run(checkpoint_id=latest.checkpoint_id)
pending = restored.get_request_info_events()
print("restored; still waiting for:", [r.data for r in pending])

final = await approval.run(responses={pending[0].request_id: True})  # the supervisor says yes
print("after approval:", final.get_outputs())

In production the folder would be durable storage (a database or blob container), and the request would reach the supervisor through a queue or a UI. The workflow code does not change.

## Part 6 · Orchestration patterns: group chat and handoff

Agent Framework ships the common multi-agent patterns as builders (lecture §8). Two of them here.

**Group chat**: participants share one conversation and a selection function picks who speaks next. Writer and critic take turns until the critic approves or the round limit is hit:

In [ ]:
from agent_framework.orchestrations import GroupChatBuilder, HandoffBuilder

writer = Agent(client=client, name="writer", instructions="Write or revise a three-sentence reply to the rider about the case. Return only the reply.")
critic = Agent(
    client=client,
    name="critic",
    instructions="Critique the latest reply in one or two bullets: accuracy, tone, no promises we cannot keep. "
    "If it is good, reply with the single word APPROVED.",
)


def round_robin(chat_state) -> str:
    return ["writer", "critic"][chat_state.current_round % 2]


def approved(conversation) -> bool:
    return any(m.author_name == "critic" and "APPROVED" in (m.text or "") for m in conversation)


chat = (
    GroupChatBuilder(participants=[writer, critic], selection_func=round_robin, max_rounds=6, intermediate_output_from="all")
    .with_termination_condition(approved)
    .build()
)
result = await chat.run(CASE)
for out in result.get_intermediate_outputs():
    for m in getattr(out, "messages", None) or [out]:
        if getattr(m, "text", ""):
            print(f"[{m.author_name}] {m.text[:220]}\n")

**Handoff**: no central plan. The current agent decides to pass control to another one, which Agent Framework exposes to the model as a tool call. Each agent keeps its own conversation history, hence `require_per_service_call_history_persistence=True`. When no agent hands off further, the workflow pauses and waits for the user.

In [ ]:
handoff_kw = {"client": client, "require_per_service_call_history_persistence": True}
front_desk = Agent(**handoff_kw, name="triage", description="Routes the case.", instructions="Hand the case to the right specialist. Do not answer yourself.")
billing_agent = Agent(**handoff_kw, name="billing", description="Money, fares and credits.", instructions="Answer the billing part using quote_fare. Be brief.", tools=[quote_fare])
ops_agent = Agent(**handoff_kw, name="operations", description="Bikes, docks and stations.", instructions="Answer the operations part using get_station. Be brief.", tools=[get_station])

handoff = (
    HandoffBuilder(participants=[front_desk, billing_agent, ops_agent])
    .with_start_agent(front_desk)
    .add_handoff(front_desk, [billing_agent, ops_agent])
    .add_handoff(billing_agent, [front_desk])
    .add_handoff(ops_agent, [front_desk])
    .build()
)
result = await handoff.run("A member rode a classic bike for 50 minutes. What does that cost?")
for out in result.get_outputs():
    for m in out if isinstance(out, list) else (getattr(out, "messages", None) or [out]):
        if getattr(m, "text", ""):
            print(f"[{m.author_name or m.role}] {m.text[:220]}")
print("\nwaiting for:", [type(e.data).__name__ for e in result.get_request_info_events()])

## Part 7 · Agents as tools, and what a team costs

The third shape keeps one agent in control: specialists become **tools** of a manager (`agent.as_tool()`). Is the team worth it? Count every model call and token, across all agents, with a **chat middleware** (it wraps each call to the model), and compare with one agent that holds both tools directly.

In [ ]:
import time

from agent_framework import ChatContext, chat_middleware

ledger: list[int] = []  # tokens of every model call, across all agents


@chat_middleware
async def count_calls(context: ChatContext, call_next) -> None:
    def record(response):
        ledger.append((response.usage_details or {}).get("total_token_count", 0))
        return response

    if context.stream:  # agents called as tools stream: read usage when the stream finishes
        context.stream_result_hooks.append(record)
    await call_next()
    if not context.stream:
        record(context.result)


billing_tool = Agent(client=client, name="billing", description="Computes fares.", instructions="Compute fares with quote_fare. Reply with the amount.", tools=[quote_fare], middleware=[count_calls])
ops_tool = Agent(client=client, name="operations", description="Checks stations.", instructions="Report station availability with get_station.", tools=[get_station], middleware=[count_calls])
manager = Agent(
    client=client,
    name="manager",
    instructions="You answer riders. Ask the billing and operations specialists for facts, then write one short reply.",
    tools=[billing_tool.as_tool(), ops_tool.as_tool()],
    middleware=[count_calls],
)
solo = Agent(
    client=client,
    name="solo",
    instructions="You answer riders. Use quote_fare and get_station for facts, then write one short reply.",
    tools=[quote_fare, get_station],
    middleware=[count_calls],
)

QUESTION = "I'm a member: what does a 50-minute classic ride cost, and are there e-bikes at LS-08?"
for agent in [manager, solo]:
    ledger.clear()
    start = time.perf_counter()
    reply = await agent.run(QUESTION)
    seconds = time.perf_counter() - start
    print(f"{agent.name:8s} {seconds:4.1f} s  model calls {len(ledger)}  tokens {sum(ledger):5d}  {reply.text[:100]!r}")

Same answer; the team made roughly three times the model calls, used more tokens, and took longer. A team pays for every hand-over: each specialist re-reads its instructions and its task, and the manager reads their replies. Split an agent only when the split buys something you can measure: least privilege (a specialist that *cannot* issue credit), a different model per role, or quality on cases a single agent gets wrong (lecture §1 and §11).

## Part 8 · Evaluating an agent with `evaluate_agent`

The harness grades your whole service. Inside the code you can grade one agent with Agent Framework's evaluation helpers: run a set of queries, then apply checks to each response. Local checks are deterministic (was a tool called, does a keyword appear); the same API accepts Foundry evaluators for model-graded criteria.

In [ ]:
from agent_framework import LocalEvaluator, evaluate_agent, keyword_check, tool_called_check

expected = domain.quote_fare("member", 50, False)["total"]
ask_agent = Agent(
    client=client,
    name="ask",
    instructions="Answer Lakeshore Ride questions. Use quote_fare for prices and get_station for stations.",
    tools=[quote_fare, get_station],
)
queries = [
    "I'm a member. What does a 50-minute classic ride cost?",
    "As a member, how much for 50 minutes on a classic bike?",
    "Member here: price of a 50 min classic ride, please.",
]
with recording(state):
    results = await evaluate_agent(
        agent=ask_agent,
        queries=queries,
        evaluators=LocalEvaluator(tool_called_check("quote_fare"), keyword_check(f"{expected:.2f}")),
    )
for r in results:
    print(f"passed {r.passed} of {r.total}; all passed: {r.all_passed}")
    for query, item in zip(queries, r.items, strict=True):
        print(f"  {item.status:5s} {[(s.name, s.passed) for s in item.scores]}  {query}")

Three phrasings of one question, two checks each. Put a set like this in `tests/` (with a real model, marked so it does not run offline) and you have a regression gate for one agent: Week 10 returns to this.

**Beyond one process.** Agent Framework agents can also be **Foundry agents** (definitions stored and versioned in your project, as in the Week 5 deep dive), talk to agents in other frameworks over **A2A**, and stream to a UI over **AG-UI** (lecture §13).

### What to remember

- A **workflow** is typed executors and edges; it runs in **supersteps**, and its events are what you log or stream.
- **Fan-out/fan-in**, **switch-case** and **multi-selection** edges express parallelism and routing in the graph, not in ad-hoc code.
- **Middleware** enforces policy in code: a tool call it refuses never runs.
- **Requests + checkpoints** let a workflow wait for a person across restarts.
- **Group chat**, **handoff** and **agents as tools** are builders, not frameworks to write; pick by who keeps control.
- A team **costs** calls and tokens: split only for a measurable benefit, and evaluate each agent like code.

**Learn more:** [Workflows](https://learn.microsoft.com/en-us/agent-framework/workflows/overview) · [Orchestrations](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/) · [Middleware](https://learn.microsoft.com/en-us/agent-framework/agents/middleware/) · [Human in the loop](https://learn.microsoft.com/en-us/agent-framework/workflows/human-in-the-loop) · [Checkpoints](https://learn.microsoft.com/en-us/agent-framework/workflows/checkpoints)